# Nik Studio - animate

**You give it pictures and a song. It gives you one finished MP4.**

Two cells. Run cell 1, then cell 2. Nothing else.

### Before you start

Make this folder in Google Drive and put your files in it:

```
My Drive / NikStudio / Input /
      Scene01.png
      Scene02.png
      Scene03.png
      song.mp3
```

Any names work — the pictures are used **in alphabetical order**, so
number them. Any one audio file is taken as the song.

The finished video comes back as:

```
My Drive / NikStudio / Output / Episode.mp4
```

### What it does

Every picture becomes a moving clip, the clips are cut to share the
length of the song exactly, and the song is laid over the top.

Set **Runtime > Change runtime type** to the best GPU you have. It reads
the card and picks its own quality settings, so there is nothing to tune.

It saves each clip to Drive as it finishes. If the session dies, run it
again — it picks up where it stopped instead of starting over.

**Honest about one thing:** the mouth moves, but it is not lip-synced to
the words. Nothing free does real lip-sync yet.


In [ ]:
# ======================================================================
# CELL 1 of 2 - the packages
# ======================================================================
#
# If Colab offers "RESTART SESSION" when this finishes, click it.
# Cell 2 depends on nothing in here, so a restart costs nothing.

!pip install -q "diffusers>=0.32" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg

print("Packages installed. Now run cell 2.")


In [ ]:
# ======================================================================
# CELL 2 of 2 - the whole thing
# ======================================================================
#
# Finds your pictures and your song in Drive, animates every picture,
# cuts the clips to share the song's length, lays the song over the top,
# and plays you the result.
#
# It depends on nothing above it, so running cells out of order or
# letting Colab restart the runtime cannot break it.

import gc
import json
import re
import subprocess
import time

from pathlib import Path

import torch

from PIL import Image


# ======================================================================
# SETTINGS - the only part worth editing
# ======================================================================

FOLDER = "/content/drive/MyDrive/NikStudio"

# What HAPPENS in the shot. Not what the picture shows - that is already
# there. One clear physical action beats three vague ones.
PROMPT = (
    "The little boy sings, his mouth opening and closing with the words, "
    "swaying from side to side and clapping his hands to the beat. He "
    "smiles and laughs. The camera stays still. Pixar style 3D "
    "animation, smooth natural motion, bright and cheerful."
)

# A picture can have its own, by file name. Anything not listed here
# uses PROMPT above.
PROMPTS = {
    # "Scene02.png": "The boy splashes the water with both hands, ...",
}

NEGATIVE = (
    "worst quality, blurry, jittery, distorted, deformed face, "
    "static image, no movement, extra limbs, watermark, text"
)

# Seconds per picture when there is no song to divide up.
FALLBACK_SECONDS = 5.0

STEPS = 50
FPS = 24

# The model makes a fixed number of frames, so a picture that has to be
# on screen longer than those frames cover has two ways to fill the time.
# Stretching the finished clip repeats frames and judders. Telling the
# model the clip is slower instead makes it generate slower movement,
# which looks natural - it just arrives as fewer frames a second.
#
# 12 is the floor. Hand drawn animation has run on twelve drawings a
# second for a century, so it is a look rather than a fault; below that
# it starts to strobe. Past the floor, stretching takes over.
MIN_FPS = 12

# The run stops when there are too few pictures for the song, because
# the clips would have to be slowed past the point of looking right
# and a paid GPU should not be spent finding that out. Set this True
# to go ahead anyway.
ALLOW_SLOW_CLIPS = False

MODEL = "Lightricks/LTX-Video"


# ======================================================================
# The card decides the quality, not you
# ======================================================================

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > pick a GPU, then run again."
    )

CARD = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9

# Compute capability 8.0 (Ampere) or newer is where bfloat16 is real.
# Do not ask torch.cuda.is_bf16_supported() - it says True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# that emulation is slower than it is worth.
MAJOR = torch.cuda.get_device_capability()[0]

DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16

# Ordinary RAM matters as much as the card here, and is the thing that
# actually killed the earlier attempts. The 9GB text encoder is unpacked
# in RAM before it ever reaches the GPU, so a big card on a small-RAM
# runtime still dies. Squeeze the text encoder to 8-bit whenever either
# one is short, not just when the card is.
try:
    import psutil

    RAM = psutil.virtual_memory().total / 1e9

except ImportError:
    # Colab ships psutil, but a notebook that dies on a missing helper
    # before it has even looked at your files is no use to anyone.
    import os

    RAM = (
        os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    )

ROOMY_CARD = VRAM >= 20
ROOMY_RAM = RAM >= 20

if ROOMY_CARD and ROOMY_RAM:
    # Room to spare: true 16:9, full precision text encoder.
    WIDTH, HEIGHT, QUANTISE, MAX_FRAMES = 1024, 576, False, 193
elif ROOMY_CARD:
    # Big card, small runtime. Full size, but the text encoder still has
    # to go in through the narrow door.
    WIDTH, HEIGHT, QUANTISE, MAX_FRAMES = 1024, 576, True, 193
elif VRAM >= 14:
    WIDTH, HEIGHT, QUANTISE, MAX_FRAMES = 704, 384, True, 121
else:
    WIDTH, HEIGHT, QUANTISE, MAX_FRAMES = 576, 320, True, 97

print(f"GPU         : {CARD} ({VRAM:.0f}GB)")
print(f"System RAM  : {RAM:.0f}GB")
print(f"Precision   : {'bfloat16' if MAJOR >= 8 else 'float16'}"
      f"{', text encoder in 8-bit' if QUANTISE else ''}")
print(f"Video size  : {WIDTH}x{HEIGHT}")
print(f"Longest clip: {MAX_FRAMES / FPS:.1f}s")


# ======================================================================
# Your files
# ======================================================================

try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

except ImportError:
    pass

root = Path(FOLDER)

INPUT = root / "Input"
OUTPUT = root / "Output"
CLIPS = OUTPUT / "Clips"

if not INPUT.exists():
    raise SystemExit(
        f"No such folder: {INPUT}\n\n"
        "Make it in Drive and put your pictures and your song in it."
    )

CLIPS.mkdir(parents=True, exist_ok=True)

def natural_key(path):
    """
    Sort "Scene2" before "Scene10".

    Plain alphabetical order puts "10" before "2", which silently
    shuffles someone's scenes. Numbers in a name are compared as numbers.
    """

    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]


PICTURES = sorted(
    (path for path in INPUT.iterdir()
     if path.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")),
    key=natural_key,
)

if not PICTURES:
    raise SystemExit(f"No pictures in {INPUT}.")

SONG = next(
    (
        path for path in sorted(INPUT.iterdir(), key=natural_key)
        if path.suffix.lower() in (".mp3", ".wav", ".m4a", ".aac", ".ogg")
    ),
    None,
)


def seconds_of(media):
    """How long an audio or video file runs, in seconds."""

    probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(media),
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(probe.stdout.strip())
    except ValueError:
        return 0.0


if SONG:
    SONG_SECONDS = seconds_of(SONG)
    SHARE = SONG_SECONDS / len(PICTURES)
    print(f"\nSong       : {SONG.name} ({SONG_SECONDS:.1f}s)")
else:
    SONG_SECONDS = 0.0
    SHARE = FALLBACK_SECONDS
    print("\nSong       : none found - using "
          f"{FALLBACK_SECONDS:.0f}s per picture")

print(f"Pictures   : {len(PICTURES)} "
      f"({', '.join(p.name for p in PICTURES)})")
print(f"Each holds : {SHARE:.1f}s")

# ======================================================================
# Everything that can go wrong, found before the GPU is touched
# ======================================================================
#
# Loading the model takes minutes and a paid GPU is charged for them, so
# nothing here is left to be discovered halfway through the run.

problems = []

print()

for picture_file in PICTURES:

    try:
        with Image.open(picture_file) as check:
            check.verify()

        with Image.open(picture_file) as check:
            shape = check.size

    except Exception:
        problems.append(
            f"{picture_file.name} will not open. Re-save it as a PNG."
        )
        continue

    print(f"  {picture_file.name:<28} {shape[0]}x{shape[1]}")

if SONG and not SONG_SECONDS:
    problems.append(
        f"{SONG.name} cannot be read. Try a plain MP3 or WAV."
    )

# How long one clip can be. At the full frame rate the model's frames
# only cover SMOOTH seconds; below that rate the same frames cover more,
# down to LONGEST. Past LONGEST the finished clip has to be stretched,
# and stretching is the one that judders.
SMOOTH = MAX_FRAMES / FPS
LONGEST = MAX_FRAMES / MIN_FPS

if SHARE > LONGEST * 1.5:

    enough = max(1, int(SONG_SECONDS / LONGEST + 0.999))

    complaint = (
        f"{len(PICTURES)} picture(s) over {SONG_SECONDS:.0f}s means each "
        f"is on screen {SHARE:.0f}s, but the longest\n"
        f"       clip this card makes is {LONGEST:.0f}s, so each would be "
        f"stretched {SHARE / LONGEST:.1f}x and judder.\n"
        f"       Use about {enough} pictures for a song this long, or set "
        f"ALLOW_SLOW_CLIPS = True above."
    )

    if ALLOW_SLOW_CLIPS:
        print(f"\n  Warning: {complaint}")
    else:
        problems.append(complaint)

elif SHARE > LONGEST:
    print(f"\n  Note: {SHARE:.1f}s a picture is past the {LONGEST:.0f}s one "
          f"clip covers, so each will be\n        stretched "
          f"{SHARE / LONGEST:.2f}x on top of running at {MIN_FPS}fps.")

elif SHARE > SMOOTH:
    print(f"\n  Note: {SHARE:.1f}s a picture is longer than the "
          f"{SMOOTH:.1f}s the model covers at {FPS}fps,\n"
          f"        so clips are generated at about "
          f"{int(MAX_FRAMES / SHARE)}fps instead - slower movement "
          f"rather than\n        repeated frames. That is the rate hand "
          f"drawn animation has always run at.")

if problems:

    raise SystemExit(
        "\n\nStopping before the GPU is used:\n\n"
        + "\n".join(f"  -  {problem}" for problem in problems)
        + "\n\nFix these and run this cell again. Nothing has been "
          "charged for."
    )

print("\n  Everything checks out.")


# ======================================================================
# The model.  Kept between runs - loading it is most of the wait
# ======================================================================

if "pipe" not in globals():

    from diffusers import LTXImageToVideoPipeline

    print("\nLoading the model. A few minutes the first time.")

    started = time.time()

    parts = {}

    if QUANTISE:

        from transformers import BitsAndBytesConfig, T5EncoderModel

        # 9GB of text encoder will not fit through a small card's RAM at
        # full size. In 8-bit it goes straight to the GPU at about 4.7GB.
        parts["text_encoder"] = T5EncoderModel.from_pretrained(
            MODEL,
            subfolder="text_encoder",
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )

    pipe = LTXImageToVideoPipeline.from_pretrained(
        MODEL, torch_dtype=DTYPE, **parts
    )

    if QUANTISE:
        # The text encoder is already on the card; move what is left.
        pipe.transformer.to("cuda")
        pipe.vae.to("cuda")
    else:
        pipe.to("cuda")

    # Decoding every frame in one piece is what runs a card out of
    # memory. Tiling decodes it in patches instead.
    pipe.vae.enable_tiling()

    print(f"Model ready in {time.time() - started:.0f}s "
          f"({torch.cuda.memory_allocated() / 1e9:.1f}GB on the card).")

else:
    print("\nModel already loaded - reusing it.")


# ======================================================================
# One picture -> one clip
# ======================================================================

def fitted(picture):
    """Cover the frame and crop the overflow, rather than squash."""

    scale = max(WIDTH / picture.width, HEIGHT / picture.height)

    picture = picture.resize(
        (round(picture.width * scale), round(picture.height * scale)),
        Image.LANCZOS,
    )

    left = (picture.width - WIDTH) // 2
    top = (picture.height - HEIGHT) // 2

    return picture.crop((left, top, left + WIDTH, top + HEIGHT))


def plan(seconds):
    """
    How to fill `seconds` with one clip.

    Returns (frames, rate, stretch): how many frames to ask the model
    for, the frame rate to generate and save them at, and how much the
    finished clip still has to be stretched afterwards.

    Frames come first, at the full frame rate, because more frames is
    always the better answer. The model needs (frames - 1) to divide by
    8 and gets worse the longer the clip, so there is a cap. Past the
    cap the frame rate comes down instead, which buys time without
    repeating a single frame. Only past MIN_FPS does stretching start.
    """

    wanted = max(25, min(int(round(seconds * FPS)), MAX_FRAMES))

    frames = ((wanted - 1) // 8) * 8 + 1

    # Comfortably within the cap: generate at the full rate.
    if frames / FPS >= seconds:
        return frames, FPS, 1.0

    # A whole number, because that is what actually gets written into
    # the file. Working the coverage out from a rate of 22.6 and then
    # saving at 23 leaves every clip a little short, and eleven clips a
    # little short is a video that ends before the song does.
    rate = max(MIN_FPS, int(frames / seconds))

    covered = frames / rate

    return frames, rate, max(1.0, seconds / covered)


def animate(picture_file, clip_file, seconds):

    frames, rate, stretch = plan(seconds)

    image = fitted(Image.open(picture_file).convert("RGB"))

    prompt = PROMPTS.get(picture_file.name, PROMPT)

    def run(count):

        return pipe(
            image=image,
            prompt=prompt,
            negative_prompt=NEGATIVE,
            width=WIDTH,
            height=HEIGHT,
            num_frames=count,
            frame_rate=int(round(rate)),
            num_inference_steps=STEPS,
            guidance_scale=3.0,
            generator=torch.Generator("cpu").manual_seed(42),
        ).frames[0]

    try:
        video = run(frames)

    except torch.cuda.OutOfMemoryError:

        gc.collect()
        torch.cuda.empty_cache()

        frames, rate, stretch = plan(seconds / 2)

        # Half the frames still has to cover the whole slot, so it is
        # stretched the rest of the way rather than end early.
        stretch = stretch * 2

        print(f"    card ran out of room - shorter clip ({frames} frames)")

        video = run(frames)

    from diffusers.utils import export_to_video

    raw = clip_file.with_name(clip_file.stem + "_raw.mp4")

    export_to_video(video, str(raw), fps=int(round(rate)))

    # Trim to exactly the slot, and stretch only if the frame rate could
    # not be lowered far enough. Freezing on the last frame would look
    # like the video broke, so the clip always fills its time.
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw),
            "-filter:v", f"setpts={stretch:.6f}*PTS",
            "-t", f"{seconds:.3f}",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "medium", "-crf", "20",
            str(clip_file),
        ],
        check=True,
    )

    raw.unlink(missing_ok=True)

    return rate, stretch


# ======================================================================
# Every picture
# ======================================================================

# A clip is reused only when the thing it was made from has not moved.
# Skipping by file name alone is what would quietly leave you with a clip
# of last week's Scene02 after you replaced the picture.
STAMPS = CLIPS / "made.json"

try:
    stamps = json.loads(STAMPS.read_text(encoding="utf-8"))
except Exception:
    stamps = {}


def stamp_for(picture_file):

    facts = picture_file.stat()

    return {
        "picture": picture_file.name,
        "bytes": facts.st_size,
        "modified": int(facts.st_mtime),
        "prompt": PROMPTS.get(picture_file.name, PROMPT),
        "size": f"{WIDTH}x{HEIGHT}",
        "seconds": round(SHARE, 2),
    }


made = []

for number, picture_file in enumerate(PICTURES, start=1):

    clip_file = CLIPS / f"{picture_file.stem}.mp4"

    label = f"[{number}/{len(PICTURES)}] {picture_file.name}"

    wanted = stamp_for(picture_file)

    finished = clip_file.exists() and clip_file.stat().st_size > 0

    if finished and stamps.get(clip_file.name) == wanted:
        print(f"{label}: already made, skipping")
        made.append(clip_file)
        continue

    if finished:
        print(f"{label}: picture or prompt changed - making it again")

    print(f"{label}: animating {SHARE:.1f}s ...")

    started = time.time()

    rate, stretch = animate(picture_file, clip_file, SHARE)

    notes = []

    if rate < FPS - 0.5:
        notes.append(f"generated at {rate:.0f}fps to fill {SHARE:.1f}s")

    if stretch > 2.0:
        notes.append(f"stretched {stretch:.1f}x - too far, add pictures")
    elif stretch > 1.01:
        notes.append(f"stretched {stretch:.2f}x")

    note = f", {', '.join(notes)}" if notes else ""

    print(f"    done in {(time.time() - started) / 60:.1f} min{note}")

    # Written after every clip, not at the end: a session that dies has
    # to leave behind an honest record of what is really finished.
    stamps[clip_file.name] = wanted

    STAMPS.write_text(json.dumps(stamps, indent=1), encoding="utf-8")

    made.append(clip_file)


# ======================================================================
# Join them, lay the song over the top
# ======================================================================

print("\nJoining the clips ...")

listing = OUTPUT / "clips.txt"

listing.write_text(
    "".join(f"file '{clip}'\n" for clip in made),
    encoding="utf-8",
)

FINAL = OUTPUT / "Episode.mp4"

command = [
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "concat", "-safe", "0", "-i", str(listing),
]

if SONG:
    command += ["-i", str(SONG), "-c:a", "aac", "-b:a", "192k", "-shortest"]

command += [
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-preset", "medium", "-crf", "20",
    "-r", str(FPS),
    str(FINAL),
]

subprocess.run(command, check=True)

listing.unlink(missing_ok=True)

print(f"\nFinished: {FINAL}")
print(f"Length  : {seconds_of(FINAL):.1f}s")

from IPython.display import Video, display

display(Video(str(FINAL), embed=True, width=min(WIDTH, 720)))

# ----------------------------------------------------------------------
# The file is in Drive, so it is already on your PC if Drive syncs there.
#
# Want a picture to do something else? Add it to PROMPTS at the top,
# delete that clip from Output/Clips, and run this cell again - the other
# clips are kept, so only the one you changed is made afresh.
# ----------------------------------------------------------------------
